# Clear Creek call forecast: quick start

Runs anywhere with HTTPS (Colab included). It fetches the call truth table hydrohistorian.com publishes, shows the seasons, runs the three baselines through the leave-one-season-out hindcast, and prints the verdict the site's gate would give.

Read `docs/TARGET.md` for what the label is.

In [ ]:
# In Colab: install from GitHub. Locally, `pip install -e .` in the repo instead.
import importlib
if importlib.util.find_spec('ccf') is None:
    %pip -q install git+https://github.com/crawforj/clear-creek-call-forecast

In [ ]:
from ccf import data, truth, hindcast, features
from ccf.models import Climatology, Persistence, SnowAnalog

manifest, weekly, daily = data.truth_table()
bands = [int(b) for b in manifest['bands']]
print(manifest['stream'], 'built', manifest['built_at'], 'data through', manifest['data_through'])
print('usable seasons:', truth.usable_seasons(weekly))
truth.season_summary(weekly)

## The label, one season
`controlling_admin_combined` is the target: the most senior applied call in force that week (smaller = more senior; empty = free river).

In [ ]:
year = truth.usable_seasons(weekly)[-1]
weekly[weekly.season_year == year][['week_index', 'week_start', 'controlling_admin_stream', 'controlling_admin_mainstem', 'controlling_admin_combined', 'is_free_river']]

In [ ]:
import matplotlib.pyplot as plt
u = weekly[weekly.usable]
piv = u.pivot(index='season_year', columns='week_index', values='controlling_admin_combined')
fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(piv.values, aspect='auto', cmap='viridis_r')
ax.set_yticks(range(len(piv.index))); ax.set_yticklabels(piv.index)
ax.set_xlabel('week of season (0 = April 1)'); ax.set_title('Controlling admin number on Clear Creek (dark = senior call; white = free river)')
plt.colorbar(im, label='admin number'); plt.show()

## The baselines, scored the way the site scores them

In [ ]:
feats = {'april1_swe': features.april1_swe()}
for cls, kw in ((Climatology, {}), (SnowAnalog, {'k': 5})):
    sk = hindcast.run(weekly, bands, cls, kw, features=feats if cls.requires else None, n_boot=500)
    print(cls.name, '->', sk['verdict'])
    print('   per band skill vs climatology:', {b: v['bss_vs_climatology'] for b, v in sk['per_band'].items()})

## Your model
Subclass `ccf.models.Model`, implement `fit` and `predict`, run it through `hindcast.run`. The harness never lets a season see itself. If the verdict starts with `PASSES`, open a PR with the report (`CONTRIBUTING.md`).

In [ ]:
swe = features.april1_swe()
summary = truth.season_summary(weekly).set_index('season_year').join(swe[['swe_pct_median']])
ax = summary.plot.scatter(x='swe_pct_median', y='weeks_under_call', figsize=(7, 5))
for y, r in summary.dropna().iterrows():
    ax.annotate(str(y), (r.swe_pct_median, r.weeks_under_call), fontsize=8)
ax.set_xlabel('April 1 SWE, % of median (3-site mean)'); ax.set_ylabel('weeks under call'); plt.show()